In [4]:
import pygame
import chess
import random

pygame.init()

ANCHO = 640
ALTO = 640
TAM = ANCHO // 8

pantalla = pygame.display.set_mode((ANCHO, ALTO))
pygame.display.set_caption("Ajedrez vs Computadora")

CLARO = (240, 217, 181)
OSCURO = (181, 136, 99)
VERDE = (100, 200, 100)
AZUL = (80, 120, 255)

fuente_piezas = pygame.font.SysFont("segoeuisymbol", 48)
fuente_texto = pygame.font.SysFont("arial", 28)

tablero = chess.Board()

PIEZAS = {
    "P": "♙", "N": "♘", "B": "♗", "R": "♖", "Q": "♕", "K": "♔",
    "p": "♟", "n": "♞", "b": "♝", "r": "♜", "q": "♛", "k": "♚"
}

seleccionada = None
movimientos_legales = []

# Humano juega con blancas
COLOR_HUMANO = chess.WHITE
COLOR_COMPUTADORA = chess.BLACK


def dibujar_tablero():
    for fila in range(8):
        for col in range(8):
            color = CLARO if (fila + col) % 2 == 0 else OSCURO
            pygame.draw.rect(pantalla, color, (col * TAM, fila * TAM, TAM, TAM))


def dibujar_piezas():
    for casilla in chess.SQUARES:
        pieza = tablero.piece_at(casilla)
        if pieza:
            col = chess.square_file(casilla)
            fila = 7 - chess.square_rank(casilla)

            texto = fuente_piezas.render(PIEZAS[pieza.symbol()], True, (0, 0, 0))
            rect = texto.get_rect(center=(col * TAM + TAM // 2, fila * TAM + TAM // 2))
            pantalla.blit(texto, rect)


def obtener_casilla(pos):
    x, y = pos
    col = x // TAM
    fila = y // TAM
    rank = 7 - fila
    return chess.square(col, rank)


def dibujar_seleccion():
    if seleccionada is not None:
        col = chess.square_file(seleccionada)
        fila = 7 - chess.square_rank(seleccionada)
        pygame.draw.rect(pantalla, VERDE, (col * TAM, fila * TAM, TAM, TAM), 5)

    for mov in movimientos_legales:
        destino = mov.to_square
        col = chess.square_file(destino)
        fila = 7 - chess.square_rank(destino)
        pygame.draw.circle(
            pantalla,
            AZUL,
            (col * TAM + TAM // 2, fila * TAM + TAM // 2),
            10
        )


def valor_pieza(pieza):
    valores = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 100
    }
    return valores.get(pieza.piece_type, 0)


def movimiento_computadora():
    movimientos = list(tablero.legal_moves)

    if not movimientos:
        return

    mejores = []

    for mov in movimientos:
        tablero.push(mov)

        if tablero.is_checkmate():
            tablero.pop()
            tablero.push(mov)
            return

        puntaje = 0

        if tablero.is_check():
            puntaje += 2

        tablero.pop()

        pieza_capturada = tablero.piece_at(mov.to_square)
        if pieza_capturada:
            puntaje += valor_pieza(pieza_capturada)

        if mov.promotion:
            puntaje += 8

        mejores.append((puntaje, mov))

    max_puntaje = max(p for p, m in mejores)
    candidatos = [m for p, m in mejores if p == max_puntaje]

    tablero.push(random.choice(candidatos))


def mostrar_estado():
    if tablero.is_checkmate():
        ganador = "Negras" if tablero.turn == chess.WHITE else "Blancas"
        mensaje = f"Jaque mate. Ganan {ganador}"
    elif tablero.is_stalemate():
        mensaje = "Empate por ahogado"
    elif tablero.is_insufficient_material():
        mensaje = "Empate por material insuficiente"
    elif tablero.is_check():
        mensaje = "Jaque"
    else:
        mensaje = "Tu turno" if tablero.turn == COLOR_HUMANO else "Computadora pensando"

    texto = fuente_texto.render(mensaje, True, (200, 0, 0))
    pantalla.blit(texto, (20, 20))


def mover_humano(casilla):
    global seleccionada, movimientos_legales

    if tablero.turn != COLOR_HUMANO or tablero.is_game_over():
        return

    if seleccionada is None:
        pieza = tablero.piece_at(casilla)

        if pieza and pieza.color == COLOR_HUMANO:
            seleccionada = casilla
            movimientos_legales = [
                mov for mov in tablero.legal_moves
                if mov.from_square == seleccionada
            ]

    else:
        movimiento = chess.Move(seleccionada, casilla)

        pieza = tablero.piece_at(seleccionada)
        if pieza and pieza.piece_type == chess.PAWN:
            rank_destino = chess.square_rank(casilla)
            if rank_destino == 0 or rank_destino == 7:
                movimiento = chess.Move(seleccionada, casilla, promotion=chess.QUEEN)

        if movimiento in tablero.legal_moves:
            tablero.push(movimiento)

        seleccionada = None
        movimientos_legales = []


ejecutando = True
reloj = pygame.time.Clock()

while ejecutando:
    reloj.tick(60)

    for evento in pygame.event.get():
        if evento.type == pygame.QUIT:
            ejecutando = False

        if evento.type == pygame.MOUSEBUTTONDOWN:
            casilla = obtener_casilla(evento.pos)
            mover_humano(casilla)

    if tablero.turn == COLOR_COMPUTADORA and not tablero.is_game_over():
        pygame.time.delay(300)
        movimiento_computadora()

    pantalla.fill((0, 0, 0))
    dibujar_tablero()
    dibujar_seleccion()
    dibujar_piezas()
    mostrar_estado()
    pygame.display.flip()

pygame.quit()